# Workstream 4: Decoder Lens Spectrum & Metrics (Matteo)

Notebook for developing, exploring, and validating the Logit Lens and associated metrics for autoregressive decoders (GPT-2, GePpeTto).

---

# Part I: Issue [L1] — Standard Logit Lens (GPT-2 & GePpeTto)

This section develops and validates the logic for the **Standard Logit Lens** (nostalgebraist, 2020) before porting it to the official module `src/ws4_lens/logit_lens.py`.

### Contract Specifications (`docs/CONTRACTS.md`):
- For $l < L$: $\text{logits}_l = W_U(\text{LN}_f(h_l))$
- For $l = L$: $\text{logits}_L = W_U(h_L)$ (in Hugging Face, $h_L$ already has the final layer normalization applied)
- **Sanity Check**: $\text{logits}_L$ must be numerically identical to `model(input_ids).logits`.

## 1. Environment Setup and Imports

In [40]:
import sys
from pathlib import Path
import torch

# Add repository 'src/' directory to sys.path to enable project imports
ROOT_DIR = Path.cwd().resolve().parent
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lib.data import load_model
from lib.registry import get_model_spec

print(f"Root dir: {ROOT_DIR}")
print(f"PyTorch version: {torch.__version__}")

Root dir: /Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens
PyTorch version: 2.15.0


## 2. Loading GPT-2
Load GPT-2 using the shared project interface `lib.data.load_model`.

In [41]:
model_spec = get_model_spec("gpt2")
print("Model spec:", model_spec)

model, tokenizer, metadata = load_model(
    checkpoint=model_spec["hf_id"],
    kind=model_spec["kind"],
    tokenizer_kwargs={"add_prefix_space": True},
)

print("\nModel loaded successfully:")
print(f"- Device: {metadata['device']}")
print(f"- Layers (L): {metadata['num_layers']}")
print(f"- Hidden size (d): {metadata['hidden_size']}")
print(f"- Vocab size (V): {metadata['vocab_size']}")

# Inspection of key Logit Lens components in GPT2LMHeadModel:
print("\nKey Logit Lens components:")
print("- ln_f:", model.transformer.ln_f)
print("- lm_head:", model.lm_head)

Model spec: {'key': 'gpt2', 'hf_id': 'gpt2', 'kind': 'decoder', 'lang': 'en', 'num_layers': 12, 'hidden_size': 768, 'dtype': 'float32', 'revision': 'main'}


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6978.48it/s]



Model loaded successfully:
- Device: mps
- Layers (L): 12
- Hidden size (d): 768
- Vocab size (V): 50257

Key Logit Lens components:
- ln_f: LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
- lm_head: Linear(in_features=768, out_features=50257, bias=False)


## 3. Implementation of `run_model`

Signature from `docs/CONTRACTS.md`:
```python
def run_model(model, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> list[torch.Tensor]
```
Executes the causal model in `eval` mode with `torch.no_grad()`, returning a list of $L + 1$ tensors of shape `[B, T, d]`.

In [42]:
def run_model(
    model: torch.nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor | None = None,
) -> list[torch.Tensor]:
    """
    Run the causal model and extract all intermediate hidden states.

    Parameters
    ----------
    model : torch.nn.Module
        Pretrained causal LM (e.g. GPT2LMHeadModel).
    input_ids : torch.Tensor
        Token index tensor of shape [B, T].
    attention_mask : torch.Tensor | None
        Attention mask of shape [B, T]. If None, defaults to all ones.

    Returns
    -------
    list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, d], from layer 0 (embeddings)
        up to layer L.
    """
    model.eval()
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = getattr(model, "device", input_ids.device)
    input_ids = input_ids.to(device)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

    # outputs.hidden_states is a tuple of L + 1 tensors of shape [B, T, d]
    return list(outputs.hidden_states)

## 4. Implementation of `logit_lens`

Signature from `docs/CONTRACTS.md`:
```python
def logit_lens(model, hidden_states: list[torch.Tensor]) -> list[torch.Tensor]
```
- For $l < L$: project intermediate hidden state through final LayerNorm $\text{LN}_f$ and unembedding head $W_U$.
- For $l = L$: apply unembedding head $W_U(h_L)$ directly (since $h_L$ in Hugging Face is already post-LN_f).
- Returns $L + 1$ tensors of shape `[B, T, V]`.

In [43]:
def _get_lens_components(model: torch.nn.Module) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Extract final layer norm (ln_f) and unembedding head (lm_head)."""
    # Final layer normalization (ln_f)
    if hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"):
        ln_f = model.transformer.ln_f
    elif hasattr(model, "ln_f"):
        ln_f = model.ln_f
    else:
        raise AttributeError("Could not find final layer normalization module (ln_f).")

    # Unembedding projection head (lm_head)
    if hasattr(model, "lm_head"):
        lm_head = model.lm_head
    elif hasattr(model, "get_output_embeddings"):
        lm_head = model.get_output_embeddings()
    else:
        raise AttributeError("Could not find output unembedding layer (lm_head).")

    return ln_f, lm_head


def logit_lens(model: torch.nn.Module, hidden_states: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Compute Standard Logit Lens logits across all layers from 0 to L.

    Parameters
    ----------
    model : torch.nn.Module
        Causal LM containing ln_f and lm_head.
    hidden_states : list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, d].

    Returns
    -------
    list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, V].
    """
    ln_f, lm_head = _get_lens_components(model)
    num_layers = len(hidden_states) - 1
    logits_list: list[torch.Tensor] = []

    for l, h_l in enumerate(hidden_states):
        if l < num_layers:
            # Intermediate layers: apply ln_f followed by lm_head
            logits_l = lm_head(ln_f(h_l))
        else:
            # Final layer L: h_L in Hugging Face is already normalized by ln_f
            logits_l = lm_head(h_l)
        logits_list.append(logits_l)

    return logits_list

## 5. Verification & Sanity Check with GPT-2

For a test prompt, we verify:
1. Shapes of hidden states are $[B, T, d]$.
2. Shapes of lens logits are $[B, T, V]$.
3. **Mandatory Sanity Check**: $\text{logits}_L$ strictly matches `model(input_ids).logits`.

In [44]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print(f"Prompt: '{prompt}'")
print(f"Token IDs shape: {input_ids.shape}")

# 1. Run model
hidden_states = run_model(model, input_ids, attention_mask)
print(f"Extracted layers count: {len(hidden_states)} (L + 1 = 13 for GPT-2)")
print(f"Shape of each layer hidden state: {hidden_states[0].shape}")

# 2. Run logit lens
lens_logits = logit_lens(model, hidden_states)
print(f"Computed lens logits count: {len(lens_logits)}")
print(f"Shape of each lens logit: {lens_logits[0].shape} (B, T, V)")

# 3. Official sanity check: Layer L vs Model Logits
with torch.no_grad():
    official_logits = model(input_ids.to(model.device), attention_mask=attention_mask.to(model.device)).logits

logits_L = lens_logits[-1]
diff = (logits_L - official_logits).abs().max().item()
is_exact_match = torch.allclose(logits_L, official_logits, atol=1e-5)

print("\n--- SANITY CHECK RESULT ---")
print(f"Max absolute difference between logits_L and model(...).logits: {diff:.2e}")
print(f"Are all values numerically equal within tolerance? {is_exact_match}")
assert is_exact_match, "ERROR: Layer L logits do not match official model logits!"
print("✅ Sanity check PASSED successfully!")

Prompt: 'The capital of France is'
Token IDs shape: torch.Size([1, 5])
Extracted layers count: 13 (L + 1 = 13 for GPT-2)
Shape of each layer hidden state: torch.Size([1, 5, 768])
Computed lens logits count: 13
Shape of each lens logit: torch.Size([1, 5, 50257]) (B, T, V)

--- SANITY CHECK RESULT ---
Max absolute difference between logits_L and model(...).logits: 0.00e+00
Are all values numerically equal within tolerance? True
✅ Sanity check PASSED successfully!


## 6. Visualizing Layer-by-Layer Prediction Evolution for GPT-2

Inspect the top-1 predicted next token at the last sequence position ($t = -1$) as representation depth increases.

In [45]:
print(f"Layer-wise next-token prediction evolution for '{prompt}':\n")
print(f"{'Layer':<8} | {'Predicted Top-1 Token':<25} | {'Logit':<10}")
print("-" * 50)

for l, logits_l in enumerate(lens_logits):
    last_pos_logits = logits_l[0, -1, :]
    top1_id = last_pos_logits.argmax().item()
    top1_token = tokenizer.decode([top1_id])
    top1_val = last_pos_logits[top1_id].item()
    print(f"Layer {l:<2}   | {repr(top1_token):<25} | {top1_val:.2f}")

Layer-wise next-token prediction evolution for 'The capital of France is':

Layer    | Predicted Top-1 Token     | Logit     
--------------------------------------------------
Layer 0    | ' destro'                 | 87.56
Layer 1    | ' not'                    | 5.53
Layer 2    | ' not'                    | 5.87
Layer 3    | ' now'                    | 4.58
Layer 4    | ' now'                    | 4.79
Layer 5    | ' now'                    | 1.57
Layer 6    | ' now'                    | -0.57
Layer 7    | ' now'                    | -3.13
Layer 8    | ' now'                    | -7.24
Layer 9    | ' now'                    | -12.71
Layer 10   | ' France'                 | -17.24
Layer 11   | ' France'                 | -55.42
Layer 12   | ' the'                    | -101.04


### 6.1 Top-5 Candidate Tokens for GPT-2 (Layer 12)
Inspect the probability distribution over the top 5 next-token candidates at the final layer.

In [46]:
# Inspect top-5 candidate tokens predicted by GPT-2 at the final layer (Layer 12)
last_logits = lens_logits[-1][0, -1, :]
probs = torch.softmax(last_logits, dim=-1)
top5 = torch.topk(probs, k=5)

print(f"Top 5 candidates for GPT-2 at Layer 12 for '{prompt}':")
for rank, (token_id, prob) in enumerate(zip(top5.indices, top5.values), 1):
    token_str = tokenizer.decode([token_id.item()])
    print(f"  #{rank}: {repr(token_str):<15} (prob: {prob.item() * 100:.2f}%)")

Top 5 candidates for GPT-2 at Layer 12 for 'The capital of France is':
  #1: ' the'          (prob: 8.79%)
  #2: ' now'          (prob: 4.55%)
  #3: ' a'            (prob: 3.77%)
  #4: ' Paris'        (prob: 3.66%)
  #5: ' France'       (prob: 3.47%)


## 7. Verification with Italian Decoder: GePpeTto

As specified in issue `[L1]`, the lens must operate identically on `LorenzoDeMattei/GePpeTto`.

In [47]:
geppetto_spec = get_model_spec("geppetto")
print("GePpeTto spec:", geppetto_spec)

model_it, tokenizer_it, metadata_it = load_model(
    checkpoint=geppetto_spec["hf_id"],
    kind=geppetto_spec["kind"],
)

prompt_it = "La capitale d'Italia è"
inputs_it = tokenizer_it(prompt_it, return_tensors="pt")

hidden_states_it = run_model(model_it, inputs_it["input_ids"], inputs_it["attention_mask"])
lens_logits_it = logit_lens(model_it, hidden_states_it)

# Sanity check on GePpeTto
with torch.no_grad():
    official_logits_it = model_it(
        inputs_it["input_ids"].to(model_it.device),
        attention_mask=inputs_it["attention_mask"].to(model_it.device)
    ).logits

diff_it = (lens_logits_it[-1] - official_logits_it).abs().max().item()
assert torch.allclose(lens_logits_it[-1], official_logits_it, atol=1e-5), "Sanity check failed on GePpeTto!"
print(f"✅ Sanity check on GePpeTto PASSED! (max diff: {diff_it:.2e})")

# Prediction inspection for GePpeTto
print(f"\nLayer-wise prediction evolution for '{prompt_it}':")
for l, logits_l in enumerate(lens_logits_it):
    top1_id = logits_l[0, -1, :].argmax().item()
    print(f"Layer {l:<2}: {repr(tokenizer_it.decode([top1_id]))}")

GePpeTto spec: {'key': 'geppetto', 'hf_id': 'LorenzoDeMattei/GePpeTto', 'kind': 'decoder', 'lang': 'it', 'num_layers': 12, 'hidden_size': 768, 'dtype': 'float32', 'revision': 'main'}


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7268.82it/s]


✅ Sanity check on GePpeTto PASSED! (max diff: 0.00e+00)

Layer-wise prediction evolution for 'La capitale d'Italia è':
Layer 0 : ' è'
Layer 1 : ' stata'
Layer 2 : ' stata'
Layer 3 : ' stata'
Layer 4 : ' stata'
Layer 5 : ' stata'
Layer 6 : ' stata'
Layer 7 : ' stata'
Layer 8 : ' stata'
Layer 9 : ' stata'
Layer 10: ' stata'
Layer 11: ' la'
Layer 12: ' la'


### 7.1 Top-5 Candidate Tokens for GePpeTto (Layer 12)
Inspect the probability distribution over the top 5 next-token candidates for the Italian model.

In [48]:
# Inspect top-5 candidate tokens predicted by GePpeTto at the final layer (Layer 12)
last_logits_it = lens_logits_it[-1][0, -1, :]
probs_it = torch.softmax(last_logits_it, dim=-1)
top5_it = torch.topk(probs_it, k=5)

print(f"Top 5 candidates for GePpeTto at Layer 12 for '{prompt_it}':")
for rank, (token_id, prob) in enumerate(zip(top5_it.indices, top5_it.values), 1):
    token_str = tokenizer_it.decode([token_id.item()])
    print(f"  #{rank}: {repr(token_str):<15} (prob: {prob.item() * 100:.2f}%)")

Top 5 candidates for GePpeTto at Layer 12 for 'La capitale d'Italia è':
  #1: ' la'           (prob: 9.43%)
  #2: ' il'           (prob: 3.39%)
  #3: ' stata'        (prob: 3.00%)
  #4: ' Milano'       (prob: 2.88%)
  #5: ' l'            (prob: 2.13%)


---

# Part II: Issue [L2] — Lens Metrics (Entropy, KL Divergence, Top-1 Accuracy)

In this section, we implement and test the three evaluation metrics specified in `docs/CONTRACTS.md` (Section 9.3 and Section 10) before porting them to `src/ws4_lens/metrics.py`:

1. **Shannon Entropy (nats)**: Measures prediction uncertainty over the vocabulary $V$ at layer $l$:
   $$H(P_t^{(l)}) = - \sum_{v \in V} P_t^{(l)}(v) \log P_t^{(l)}(v)$$
2. **Kullback-Leibler (KL) Divergence to Final Layer $L$ (nats)**: Measures distributional distance between layer $l$ and the model's final prediction:
   $$D_{\text{KL}}(P_t^{(L)} \parallel P_t^{(l)}) = \sum_{v \in V} P_t^{(L)}(v) \left(\log P_t^{(L)}(v) - \log P_t^{(l)}(v)\right)$$
3. **Top-1 Accuracy**: Fraction of sequence positions where the predicted top-1 token matches the ground-truth target:
   $$\text{Acc}_{\text{Top-1}}^{(l)} = \frac{1}{N} \sum_{t=1}^N \mathbb{I}\left(\arg\max_{v \in V} P_t^{(l)}(v) = y_t\right)$$

## 1. Implementation of Lens Metrics

In [49]:
def entropy(logits: torch.Tensor) -> torch.Tensor:
    """
    Compute Shannon entropy in nats from logits.
    Cast logits to float32 for numerical stability.
    entropy = -(logp.exp() * logp).sum(-1)
    """
    logits = logits.float()
    logp = torch.log_softmax(logits, dim=-1)
    ent = -(logp.exp() * logp).sum(dim=-1)
    return torch.clamp(ent, min=0.0)


def kl_to_final(logits_l: torch.Tensor, logits_L: torch.Tensor) -> torch.Tensor:
    """
    Compute Kullback-Leibler divergence KL(P_L || P_l) in nats.
    Cast logits to float32 for numerical stability.
    kl_to_final = (logp_L.exp() * (logp_L - logp_l)).sum(-1), clamped at 0 from below.
    """
    logits_l = logits_l.float()
    logits_L = logits_L.float()
    logp_l = torch.log_softmax(logits_l, dim=-1)
    logp_L = torch.log_softmax(logits_L, dim=-1)
    kl = (logp_L.exp() * (logp_L - logp_l)).sum(dim=-1)
    return torch.clamp(kl, min=0.0)


def top1_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    """
    Compute Top-1 prediction accuracy over valid target positions (targets >= 0).
    Positions where targets < 0 (such as -100 for ignored/padding tokens) are excluded.
    Automatically aligns targets to the device of logits.
    """
    targets = targets.to(logits.device)
    preds = logits.argmax(dim=-1)
    mask = targets >= 0
    if not mask.any():
        return 0.0
    return float((preds[mask] == targets[mask]).float().mean().item())


## 2. Verification of Mathematical Properties on Synthetic Data
Check known analytical values:
- Uniform distribution over $V$: entropy must equal $\ln(V)$.
- One-hot distribution: entropy must equal $0.0$.
- KL divergence to self: $D_{\text{KL}}(P_L \parallel P_L) = 0.0$.
- Top-1 accuracy with masked positions (targets = -100).

In [50]:
import math

vocab_size = 4
# 1. Uniform distribution: logits = all zeros -> P = [1/4, 1/4, 1/4, 1/4]
uniform_logits = torch.zeros(vocab_size)
ent_uniform = entropy(uniform_logits).item()
expected_uniform = math.log(vocab_size)
assert abs(ent_uniform - expected_uniform) < 1e-6, f"Expected {expected_uniform}, got {ent_uniform}"
print(f"✅ Uniform entropy: {ent_uniform:.4f} nats == ln({vocab_size}) ({expected_uniform:.4f})")

# 2. Deterministic distribution: one huge logit -> P = [1, 0, 0, 0]
one_hot_logits = torch.tensor([100.0, 0.0, 0.0, 0.0])
ent_one_hot = entropy(one_hot_logits).item()
assert abs(ent_one_hot - 0.0) < 1e-6
print(f"✅ Deterministic entropy: {ent_one_hot:.4f} nats == 0.0000")

# 3. Self KL: KL(P_L || P_L) must be exactly 0.0
random_logits = torch.randn(2, 5, vocab_size)
kl_self = kl_to_final(random_logits, random_logits)
assert torch.allclose(kl_self, torch.zeros_like(kl_self), atol=1e-6)
print(f"✅ Self KL divergence KL(P_L || P_L): max = {kl_self.max().item():.2e} == 0.0")

# 4. Non-negativity of KL: KL(P_L || P_l) >= 0
random_logits_l = torch.randn(2, 5, vocab_size)
kl_diff = kl_to_final(random_logits_l, random_logits)
assert (kl_diff >= 0.0).all()
print(f"✅ KL non-negativity: min = {kl_diff.min().item():.4f} >= 0.0")

# 5. Top-1 Accuracy with ignored position (-100)
test_logits = torch.tensor([[10.0, 0.0], [0.0, 10.0], [10.0, 0.0], [10.0, 0.0]])
test_targets = torch.tensor([0, 1, 1, -100])  # targets: [0 (correct), 1 (correct), 1 (wrong), -100 (ignored)]
acc = top1_accuracy(test_logits, test_targets)
assert abs(acc - 2.0 / 3.0) < 1e-5
print(f"✅ Top-1 Accuracy with -100 ignored: {acc:.4f} == 0.6667")

# 6. Half precision (float16) test: cast to float32 check
fp16_logits = torch.randn(2, 3, vocab_size).half()
ent_fp16 = entropy(fp16_logits)
assert ent_fp16.dtype == torch.float32
print(f"✅ Float16 auto-casting: output dtype is {ent_fp16.dtype}")


✅ Uniform entropy: 1.3863 nats == ln(4) (1.3863)
✅ Deterministic entropy: 0.0000 nats == 0.0000
✅ Self KL divergence KL(P_L || P_L): max = 0.00e+00 == 0.0
✅ KL non-negativity: min = 0.0597 >= 0.0
✅ Top-1 Accuracy with -100 ignored: 0.6667 == 0.6667
✅ Float16 auto-casting: output dtype is torch.float32


## 3. Layer-by-Layer Trajectories on GPT-2
Inspect how Entropy and KL Divergence evolve across all 13 layers ($0 \dots 12$) for the test sentence.

In [51]:
logits_final = lens_logits[-1]  # Layer 12 logits [B, T, V]

print(f"Layer-wise Metrics for GPT-2 on '{prompt}':\n")
print(f"{'Layer':<8} | {'Top-1 Token':<20} | {'Top-1 Prob':<12} | {'Entropy (nats)':<16} | {'KL to Final (nats)':<20}")
print("-" * 84)

for layer_idx, logits_l in enumerate(lens_logits):
    # Focus on the last token position predicting the next word
    pos_l = logits_l[0, -1, :]
    pos_L = logits_final[0, -1, :]
    
    # Metrics
    ent_val = entropy(pos_l).item()
    kl_val = kl_to_final(pos_l, pos_L).item()
    
    # Top-1 prediction and prob
    prob_l = torch.softmax(pos_l, dim=-1)
    top1_id = pos_l.argmax().item()
    top1_tok = tokenizer.decode([top1_id])
    top1_prob = prob_l[top1_id].item() * 100
    
    print(f"Layer {layer_idx:<2}   | {repr(top1_tok):<20} | {top1_prob:>6.2f}%     | {ent_val:>8.4f}         | {kl_val:>10.4f}")

Layer-wise Metrics for GPT-2 on 'The capital of France is':

Layer    | Top-1 Token          | Top-1 Prob   | Entropy (nats)   | KL to Final (nats)  
------------------------------------------------------------------------------------
Layer 0    | ' destro'            |  53.27%     |   1.2448         |    26.8211
Layer 1    | ' not'               |  33.21%     |   3.5704         |     3.5505
Layer 2    | ' not'               |  28.19%     |   2.7691         |     4.4942
Layer 3    | ' now'               |  35.58%     |   2.5202         |     4.5078
Layer 4    | ' now'               |  50.08%     |   2.0764         |     4.4109
Layer 5    | ' now'               |  50.66%     |   2.2676         |     3.7237
Layer 6    | ' now'               |  44.98%     |   2.1203         |     4.7028
Layer 7    | ' now'               |  60.65%     |   1.9446         |     3.2501
Layer 8    | ' now'               |  66.25%     |   1.6838         |     3.4352
Layer 9    | ' now'               |  34.84%  

## 4. Layer-by-Layer Trajectories on GePpeTto
Inspect Entropy and KL Divergence trajectories for the Italian decoder model.

In [52]:
logits_final_it = lens_logits_it[-1]

print(f"Layer-wise Metrics for GePpeTto on '{prompt_it}':\n")
print(f"{'Layer':<8} | {'Top-1 Token':<20} | {'Top-1 Prob':<12} | {'Entropy (nats)':<16} | {'KL to Final (nats)':<20}")
print("-" * 84)

for layer_idx, logits_l in enumerate(lens_logits_it):
    pos_l = logits_l[0, -1, :]
    pos_L = logits_final_it[0, -1, :]
    
    ent_val = entropy(pos_l).item()
    kl_val = kl_to_final(pos_l, pos_L).item()
    
    prob_l = torch.softmax(pos_l, dim=-1)
    top1_id = pos_l.argmax().item()
    top1_tok = tokenizer_it.decode([top1_id])
    top1_prob = prob_l[top1_id].item() * 100
    
    print(f"Layer {layer_idx:<2}   | {repr(top1_tok):<20} | {top1_prob:>6.2f}%     | {ent_val:>8.4f}         | {kl_val:>10.4f}")

Layer-wise Metrics for GePpeTto on 'La capitale d'Italia è':

Layer    | Top-1 Token          | Top-1 Prob   | Entropy (nats)   | KL to Final (nats)  
------------------------------------------------------------------------------------
Layer 0    | ' è'                 | 100.00%     |   0.0000         |    48.7073
Layer 1    | ' stata'             |  30.67%     |   2.1919         |     5.1153
Layer 2    | ' stata'             |  49.29%     |   1.7055         |     5.2860
Layer 3    | ' stata'             |  46.43%     |   1.5919         |     5.5400
Layer 4    | ' stata'             |  47.65%     |   1.4869         |     6.1037
Layer 5    | ' stata'             |  47.42%     |   1.3338         |     6.4428
Layer 6    | ' stata'             |  55.96%     |   1.1668         |     6.1111
Layer 7    | ' stata'             |  48.07%     |   1.5130         |     4.6981
Layer 8    | ' stata'             |  66.06%     |   1.3959         |     3.7207
Layer 9    | ' stata'             |  67.99% 

## 5. Top-1 Accuracy on Next-Token Prediction Task
Evaluate next-token prediction accuracy on the prompt tokens: position $t$ predicts target token at position $t+1$.

In [53]:
# Next-token prediction: position t in [0 .. T-2] predicts target token at position t + 1
# Targets: input_ids[:, 1:], ensure targets match logits device
eval_targets = input_ids[:, 1:].to(lens_logits[0].device)

print(f"Prompt: '{prompt}'")
print(f"Sequence length: {input_ids.shape[1]} tokens")
print(f"Evaluated positions count: {eval_targets.shape[1]}\n")

# Inspect token-by-token comparison at Layer 12
final_preds = lens_logits[-1][:, :-1, :].argmax(dim=-1)[0]
gold_targets = eval_targets[0]
print("Token-by-token Next-Token Prediction at Final Layer 12:")
for t, (pred_id, gold_id) in enumerate(zip(final_preds, gold_targets)):
    context_tok = tokenizer.decode([input_ids[0, t].item()])
    pred_tok = tokenizer.decode([pred_id.item()])
    gold_tok = tokenizer.decode([gold_id.item()])
    is_correct = "✅" if pred_id == gold_id else "❌"
    print(f"  Pos {t} after {repr(context_tok):<12} -> Predicted: {repr(pred_tok):<12} | Gold: {repr(gold_tok):<12} {is_correct}")

print("\nLayer-wise Top-1 Next-Token Accuracy across all layers:")
for layer_idx, logits_l in enumerate(lens_logits):
    # Logits at positions 0 .. T-2 predicting positions 1 .. T-1
    pred_logits = logits_l[:, :-1, :]
    acc = top1_accuracy(pred_logits, eval_targets)
    print(f"Layer {layer_idx:<2}: Top-1 Accuracy = {acc * 100:>5.1f}%")


Prompt: 'The capital of France is'
Sequence length: 5 tokens
Evaluated positions count: 4

Token-by-token Next-Token Prediction at Final Layer 12:
  Pos 0 after ' The'       -> Predicted: '\n'         | Gold: ' capital'   ❌
  Pos 1 after ' capital'   -> Predicted: ' of'        | Gold: ' of'        ✅
  Pos 2 after ' of'        -> Predicted: ' the'       | Gold: ' France'    ❌
  Pos 3 after ' France'    -> Predicted: ','          | Gold: ' is'        ❌

Layer-wise Top-1 Next-Token Accuracy across all layers:
Layer 0 : Top-1 Accuracy =   0.0%
Layer 1 : Top-1 Accuracy =   0.0%
Layer 2 : Top-1 Accuracy =   0.0%
Layer 3 : Top-1 Accuracy =   0.0%
Layer 4 : Top-1 Accuracy =   0.0%
Layer 5 : Top-1 Accuracy =   0.0%
Layer 6 : Top-1 Accuracy =   0.0%
Layer 7 : Top-1 Accuracy =   0.0%
Layer 8 : Top-1 Accuracy =   0.0%
Layer 9 : Top-1 Accuracy =   0.0%
Layer 10: Top-1 Accuracy =   0.0%
Layer 11: Top-1 Accuracy =   0.0%
Layer 12: Top-1 Accuracy =  25.0%


---

# Part III: Issue [L3] — Tuned Lens (Architecture, Official Checkpoint & WikiText Evaluation)

In this section, we implement and evaluate the **Tuned Lens** (Belrose et al., 2023) according to `docs/CONTRACTS.md` (Sections 9.1, 9.2, 9.5, and 10) and the Issue [L3] specifications:

1. **Architecture & Zero-Init Equivalence**: Per-layer affine translators $\tilde{h}_l = h_l + A_l h_l + b_l$ initialized to zero ($A_l = 0, b_l = 0$) such that it strictly matches the Standard Logit Lens at initialization.
2. **Official Pretrained Lens**: `load_official_tuned_lens(model) -> object` pulling the official checkpoint from `AlignmentResearch/tuned-lens` (`lens_resource_id="gpt2"`).
3. **Polymorphic Wrapper**: `tuned_lens(model, translators, hidden_states)` accepting either custom translators or the official `TunedLens`, returning $L + 1$ logit tensors where layer $L$ equals model logits.
4. **Quantitative Benchmark on WikiText**: 50 non-overlapping windows of 256 tokens (CONTRACTS.md Section 9.1), comparing mean KL divergence to layer $L$ between Logit Lens and Tuned Lens.

## 1. Implementation of `TunedLensTranslators`, `load_official_tuned_lens`, and `tuned_lens`

In [54]:
import torch
import torch.nn as nn
from typing import Any


class TunedLensTranslators(nn.Module):
    """
    Learned per-layer affine translators for the Tuned Lens (Belrose et al., 2023).
    
    Each translator computes:
        h_tilde = h + A_l h + b_l
    Initialized to zero (A_l = 0, b_l = 0) so that it starts as an exact identity map.
    """

    def __init__(self, num_layers: int, hidden_size: int):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        self.translators = nn.ModuleList([
            nn.Linear(hidden_size, hidden_size)
            for _ in range(num_layers)
        ])
        
        for linear in self.translators:
            nn.init.zeros_(linear.weight)
            nn.init.zeros_(linear.bias)

    def forward(self, layer_idx: int, h: torch.Tensor) -> torch.Tensor:
        """Apply translator for layer_idx: h + linear(h). Returns h unchanged for layer >= num_layers."""
        if layer_idx < self.num_layers:
            return h + self.translators[layer_idx](h)
        return h

    def __getitem__(self, idx: int) -> nn.Linear:
        return self.translators[idx]

    def __len__(self) -> int:
        return self.num_layers


def load_official_tuned_lens(model: nn.Module) -> Any:
    """
    Load the official pretrained Tuned Lens for GPT-2 (Belrose et al.).
    
    Parameters
    ----------
    model : nn.Module
        Underlying Hugging Face decoder model.
        
    Returns
    -------
    object
        Official TunedLens object placed on the model's device.
    """
    from tuned_lens.nn.lenses import TunedLens
    lens = TunedLens.from_model_and_pretrained(model, lens_resource_id="gpt2")
    return lens.to(model.device)


def tuned_lens(
    model: nn.Module,
    translators: Any,
    hidden_states: list[torch.Tensor],
) -> list[torch.Tensor]:
    """
    Compute tuned lens logits across all layers 0 .. L.
    
    Parameters
    ----------
    model : nn.Module
        Autoregressive transformer decoder (e.g. GPT-2, GePpeTto).
    translators : TunedLensTranslators | object
        Learned translators (either custom TunedLensTranslators or official TunedLens).
    hidden_states : list[torch.Tensor]
        List of L + 1 hidden states [B, T, d] from run_model.
        
    Returns
    -------
    list[torch.Tensor]
        List of L + 1 logit tensors [B, T, V], where layer L equals model logits.
    """
    ln_f, lm_head = _get_lens_components(model)
    num_layers = len(hidden_states) - 1  # L
    is_official_lens = hasattr(translators, "layer_translators")

    logits = []
    for layer_idx, h in enumerate(hidden_states):
        if layer_idx < num_layers:
            if is_official_lens:
                # Official TunedLens applies translator, LayerNorm, and unembedding
                logits_l = translators(h, layer_idx)
            else:
                # Custom TunedLensTranslators: h_tilde = h + A_l h + b_l
                h_tilde = translators(layer_idx, h)
                normed = ln_f(h_tilde)
                logits_l = lm_head(normed)
        else:
            # Layer L: identity convention (bit-for-bit equal to model.logits)
            logits_l = lm_head(h)
        logits.append(logits_l)

    return logits


## 2. Verification of Zero-Initialization Equivalence to Logit Lens
With $A_l = 0$ and $b_l = 0$, `tuned_lens` must produce outputs numerically identical to `logit_lens`.

In [55]:
# Instantiate zero-initialized translators for GPT-2 (12 layers, hidden_size = 768)
translators_zero = TunedLensTranslators(num_layers=12, hidden_size=768).to(model.device)

# Run tuned lens with zero-initialized translators
tuned_logits_zero = tuned_lens(model, translators_zero, hidden_states)

print(f"Number of layers returned by tuned_lens: {len(tuned_logits_zero)}")

# Compare each layer with logit_lens
max_diffs = []
for layer_idx in range(len(lens_logits)):
    diff = (lens_logits[layer_idx] - tuned_logits_zero[layer_idx]).abs().max().item()
    max_diffs.append(diff)

print(f"Max absolute difference across all layers (Logit Lens vs Zero-Init Tuned Lens): {max(max_diffs):.2e}")
assert max(max_diffs) < 1e-5, f"Expected zero difference, got {max(max_diffs)}"
print("✅ Verification passed: Zero-initialized Tuned Lens strictly equals Standard Logit Lens!")


Number of layers returned by tuned_lens: 13
Max absolute difference across all layers (Logit Lens vs Zero-Init Tuned Lens): 0.00e+00
✅ Verification passed: Zero-initialized Tuned Lens strictly equals Standard Logit Lens!


## 3. Loading and Inspecting the Official TunedLens for GPT-2
Here we load the official lens using `load_official_tuned_lens(model)` and inspect the requirements specified in Issue [L3]:
- **Hidden-state indexing**: It expects layer indices $0 \dots L-1$ (layers $0 \dots 11$, length 12). Index 12 is out of range.
- **LayerNorm & Unembedding**: Calling `lens(h_l, l)` applies the affine translator, LayerNorm, and unembedding, returning vocabulary logits `[B, T, V]`.
- **Wrapping**: `tuned_lens(model, official_lens, hidden_states)` wraps it into $L + 1$ tensors with layer $L$ matching model logits.

In [56]:
official_lens = load_official_tuned_lens(model)
print(f"Loaded official lens object: {type(official_lens)}")
print(f"Number of translators in official lens: {len(official_lens)}")

# Check indexing: layers 0 .. 11 valid; layer 12 raises IndexError
try:
    _ = official_lens(hidden_states[12], 12)
    print("Layer 12 exists in official lens.")
except IndexError as e:
    print(f"Confirmed indexing: layer 12 raises IndexError ({e}). Official lens covers layers 0 .. L-1.")

# Verify output dimensions and device
out0 = official_lens(hidden_states[0], 0)
print(f"Output shape of official_lens(h_0, 0): {out0.shape} (applies ln_f and unembedding to vocab size {out0.shape[-1]})")

# Verify tuned_lens wrapper produces 13 layers with layer 12 matching model logits
official_logits = tuned_lens(model, official_lens, hidden_states)
print(f"tuned_lens wrapper returns {len(official_logits)} layers.")
diff_L = (official_logits[-1] - lens_logits[-1]).abs().max().item()
print(f"Max difference at layer L (Tuned Lens vs Model Logits): {diff_L:.2e}")
assert diff_L == 0.0, "Layer L must identically equal model logits!"
print("✅ Verification passed: tuned_lens wrapper adheres strictly to CONTRACTS.md conventions.")


Loaded official lens object: <class 'tuned_lens.nn.lenses.TunedLens'>
Number of translators in official lens: 12
Confirmed indexing: layer 12 raises IndexError (index 12 is out of range). Official lens covers layers 0 .. L-1.
Output shape of official_lens(h_0, 0): torch.Size([1, 5, 50257]) (applies ln_f and unembedding to vocab size 50257)
tuned_lens wrapper returns 13 layers.
Max difference at layer L (Tuned Lens vs Model Logits): 0.00e+00
✅ Verification passed: tuned_lens wrapper adheres strictly to CONTRACTS.md conventions.


## 4. Comparing Standard Logit Lens vs Tuned Lens on Test Prompt
Compare layer-wise predictions, probabilities, and next-token accuracy on prompt `"The capital of France is"`.

In [57]:
print(f"Layer-wise Comparison: Logit Lens vs Official Tuned Lens on '{prompt}':\n")
print(f"{'Layer':<7} | {'Logit Lens Top-1':<18} | {'Prob':<7} | {'Tuned Lens Top-1':<18} | {'Prob':<7} | {'H(Logit)':<8} | {'H(Tuned)':<8}")
print("-" * 88)

for layer_idx in range(13):
    pos_logit = lens_logits[layer_idx][0, -1, :]
    pos_tuned = official_logits[layer_idx][0, -1, :]
    
    p_logit = torch.softmax(pos_logit, dim=-1)
    p_tuned = torch.softmax(pos_tuned, dim=-1)
    
    id_logit = pos_logit.argmax().item()
    id_tuned = pos_tuned.argmax().item()
    
    tok_logit = tokenizer.decode([id_logit])
    tok_tuned = tokenizer.decode([id_tuned])
    
    prob_logit = p_logit[id_logit].item() * 100
    prob_tuned = p_tuned[id_tuned].item() * 100
    
    h_logit = entropy(pos_logit).item()
    h_tuned = entropy(pos_tuned).item()
    
    print(f"Layer {layer_idx:<2} | {repr(tok_logit):<18} | {prob_logit:>5.1f}% | {repr(tok_tuned):<18} | {prob_tuned:>5.1f}% | {h_logit:>7.3f}  | {h_tuned:>7.3f}")

print("\nDetailed token prediction at Position 1 (after ' capital' -> Gold target: ' of'):")
for layer_idx in range(13):
    tok_l = tokenizer.decode([lens_logits[layer_idx][0, 1].argmax().item()])
    tok_t = tokenizer.decode([official_logits[layer_idx][0, 1].argmax().item()])
    ok_l = "✅" if tok_l == " of" else "❌"
    ok_t = "✅" if tok_t == " of" else "❌"
    print(f"  Layer {layer_idx:<2}: Logit Lens -> {repr(tok_l):<12} {ok_l} | Tuned Lens -> {repr(tok_t):<12} {ok_t}")


Layer-wise Comparison: Logit Lens vs Official Tuned Lens on 'The capital of France is':

Layer   | Logit Lens Top-1   | Prob    | Tuned Lens Top-1   | Prob    | H(Logit) | H(Tuned)
----------------------------------------------------------------------------------------
Layer 0  | ' destro'          |  53.3% | ' is'              |   5.6% |   1.245  |   6.403
Layer 1  | ' not'             |  33.2% | ' a'               |   6.1% |   3.570  |   7.248
Layer 2  | ' not'             |  28.2% | ' a'               |   6.4% |   2.769  |   6.729
Layer 3  | ' now'             |  35.6% | ' a'               |   5.5% |   2.520  |   6.816
Layer 4  | ' now'             |  50.1% | ' the'             |   6.8% |   2.076  |   6.562
Layer 5  | ' now'             |  50.7% | ' the'             |   6.7% |   2.268  |   7.010
Layer 6  | ' now'             |  45.0% | ' now'             |   8.1% |   2.120  |   6.406
Layer 7  | ' now'             |  60.6% | ' the'             |   7.4% |   1.945  |   7.050
Layer 8  |

## 5. Evaluation on 50 WikiText Test Windows (CONTRACTS.md Section 9.1)

According to `docs/CONTRACTS.md` (Section 9.1):
> *On WikiText the text is concatenated and cut into non-overlapping windows of 256 tokens; use at most 200 windows (about 50k positions).*

Here we benchmark Logit Lens vs Tuned Lens on **50 non-overlapping windows of 256 tokens** ($12{,}750$ evaluated next-token positions), computing the mean KL divergence to the final layer $L$ per layer to verify the acceptance criterion:
> *Table in the PR: per-layer KL of logit vs tuned lens; tuned lens is lower in most layers before L.*

In [58]:
from datasets import load_dataset

print("Loading WikiText test split...")
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")
all_text = "\n".join([x["text"] for x in ds if x["text"].strip()])

# Tokenize full continuous stream
tokens = tokenizer(all_text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
print(f"Total tokens available in WikiText test: {len(tokens):,}")

# Slice 50 non-overlapping windows of 256 tokens
n_windows = 50
window_size = 256
windows = [tokens[i * window_size : (i + 1) * window_size] for i in range(n_windows)]

num_layers = 12
logit_kl_sums = torch.zeros(num_layers + 1)
tuned_kl_sums = torch.zeros(num_layers + 1)
total_positions = 0

print(f"Evaluating {n_windows} windows of length {window_size}...")
with torch.no_grad():
    for w_idx, win in enumerate(windows):
        input_ids = win.unsqueeze(0).to(model.device)
        attention_mask = torch.ones_like(input_ids)
        
        hidden_states = run_model(model, input_ids, attention_mask)
        logits_l = logit_lens(model, hidden_states)
        logits_t = tuned_lens(model, official_lens, hidden_states)
        
        # Positions 0 .. T-2 predict 1 .. T-1 (last position skipped per Section 9.1)
        pred_l = [l[:, :-1, :] for l in logits_l]
        pred_t = [l[:, :-1, :] for l in logits_t]
        
        final_l = pred_l[-1]
        final_t = pred_t[-1]
        
        total_positions += pred_l[0].shape[1]
        
        for layer in range(num_layers + 1):
            logit_kl_sums[layer] += kl_to_final(pred_l[layer], final_l).sum().item()
            tuned_kl_sums[layer] += kl_to_final(pred_t[layer], final_t).sum().item()

mean_kl_logit = logit_kl_sums / total_positions
mean_kl_tuned = tuned_kl_sums / total_positions

print("\n" + "=" * 75)
print(f"Mean KL Divergence to Final Layer L across {n_windows} WikiText Windows ({total_positions} positions):")
print(f"{'Layer':<7} | {'Logit Lens KL (nats)':<22} | {'Tuned Lens KL (nats)':<22} | {'Reduction (%)':<14}")
print("-" * 75)
for layer in range(num_layers + 1):
    kl_l = mean_kl_logit[layer].item()
    kl_t = mean_kl_tuned[layer].item()
    red = ((kl_l - kl_t) / kl_l * 100) if kl_l > 0 else 0.0
    print(f"Layer {layer:<2} | {kl_l:>20.4f}   | {kl_t:>20.4f}   | {red:>11.1f}%")
print("=" * 75)


Loading WikiText test split...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (283287 > 1024). Running this sequence through the model will result in indexing errors


Total tokens available in WikiText test: 283,287
Evaluating 50 windows of length 256...

Mean KL Divergence to Final Layer L across 50 WikiText Windows (12750 positions):
Layer   | Logit Lens KL (nats)   | Tuned Lens KL (nats)   | Reduction (%) 
---------------------------------------------------------------------------
Layer 0  |              49.8199   |               5.2776   |        89.4%
Layer 1  |               7.5848   |               3.0899   |        59.3%
Layer 2  |               6.5837   |               2.8910   |        56.1%
Layer 3  |               6.2609   |               2.7202   |        56.6%
Layer 4  |               6.1280   |               2.5597   |        58.2%
Layer 5  |               5.8950   |               2.4033   |        59.2%
Layer 6  |               5.2418   |               1.9453   |        62.9%
Layer 7  |               4.8314   |               1.6356   |        66.1%
Layer 8  |               4.0533   |               1.1692   |        71.2%
Layer 9  |  